In [1]:
import json
from datetime import date, timedelta

import pandas as pd
import requests
from bs4 import BeautifulSoup

from tqdm.auto import tqdm

In [2]:
url = 'https://www.europeansleeper.eu/'

In [3]:
r = requests.get(url)

In [4]:
r.status_code

200

In [5]:
soup = BeautifulSoup(r.content, 'html.parser')

In [6]:
soup

<!DOCTYPE html>

<html lang="en">
<head>
<meta charset="utf-8"/>
<!-- Google Tag Manager -->
<script>(function(w,d,s,l,i){w[l]=w[l]||[];w[l].push({'gtm.start':
new Date().getTime(),event:'gtm.js'});var f=d.getElementsByTagName(s)[0],
j=d.createElement(s),dl=l!='dataLayer'?'&l='+l:'';j.async=true;j.src=
'https://analytics.europeansleeper.eu/gtm.js?id='+i+dl;f.parentNode.insertBefore(j,f);
})(window,document,'script','dataLayer','GTM-TS9JM63');</script>
<!-- End Google Tag Manager -->
<meta content="summary_large_image" name="twitter:card"/>
<meta content="@EuropeanSlpr" name="twitter:creator"/>
<meta content="website" property="og:type"/>
<meta content="en_US" property="og:locale"/>
<meta content="European Sleeper" property="og:site_name"/>
<meta content="Discover Europe by Night Train with European Sleeper | European Sleeper" property="og:title"/>
<meta content="Travel Europe in style with our comfortable and eco-friendly night trains. Your next adventure starts here!" property="og:des

In [7]:
constants_script = soup.find('script', id='booking-search-props')
data = json.loads(constants_script.string.strip())

In [11]:
data

{'constants': {'settings': {'bookingFlowState': 2,
   'bookingFlowEnabled': True,
   'bookingFlowMessageType': 0,
   'maxReservationDate': '2026-11-08',
   'bicycleReservationDates': [{'start': '2024-04-04', 'end': '2024-09-30'},
    {'start': '2025-06-04', 'end': '2025-08-31'},
    {'start': '2026-09-01', 'end': '2026-10-31'}]},
  'domesticJourneyDisabledCountries': ['84', '88', '54', '87'],
  'stations': [{'country': '88',
    'countryName': 'Belgium',
    'stations': [{'id': '8800104',
      'name': 'Bruxelles Midi/Brussel Zuid',
      'nsName': None,
      'uicCode': '8814001',
      'countryId': '88',
      'country': {'id': '88',
       'name': 'Belgium',
       'railwayCompanyAbbreviation': 'SNCB'},
      'isPopular': True},
     {'id': '8800210',
      'name': 'Antwerpen Centraal',
      'nsName': None,
      'uicCode': '8821006',
      'countryId': '88',
      'country': {'id': '88',
       'name': 'Belgium',
       'railwayCompanyAbbreviation': 'SNCB'},
      'isPopular': Fal

In [9]:
stations = data['constants']['stations']
stations = {}
for country in data['constants']['stations']:
    for station in country['stations']:
        stations[station['id']] = station['name']
stations

{'8800104': 'Bruxelles Midi/Brussel Zuid',
 '8800210': 'Antwerpen Centraal',
 '8700015': 'Paris-Nord',
 '8400526': 'Roosendaal',
 '8400530': 'Rotterdam Centraal',
 '8400280': 'Den Haag HS',
 '8400058': 'Amsterdam Centraal',
 '8400055': 'Amersfoort Centraal',
 '8400173': 'Deventer',
 '8020401': 'Hamburg Harburg',
 '8010110': 'Berlin Ostbahnhof',
 '8010100': 'Berlin Hbf',
 '8001305': 'Dresden Hbf',
 '8001311': 'Bad Schandau',
 '5455659': 'Děčín hl.n.',
 '5453179': 'Ústí nad Labem hl.n.',
 '5457076': 'Praha hl.n. (main station)'}

In [10]:
routes = data['constants']['routes']

for route in routes:
    print(route['id'], route['from'])
    for station_id in route['stations']:
        print(stations[station_id])

77ab1c5a-ea0b-4634-7cdd-08db0daabe3f 0001-01-01T00:00:00
Bruxelles Midi/Brussel Zuid
Antwerpen Centraal
Roosendaal
Rotterdam Centraal
Den Haag HS
Amsterdam Centraal
Amersfoort Centraal
Deventer
Berlin Hbf
Berlin Ostbahnhof
Dresden Hbf
Bad Schandau
Děčín hl.n.
Ústí nad Labem hl.n.
Praha hl.n. (main station)
c181e609-05b7-456a-bab6-4203dc2c3402 2026-03-26T00:00:00
Paris-Nord
Bruxelles Midi/Brussel Zuid
Hamburg Harburg
Berlin Ostbahnhof
Berlin Hbf


In [46]:
train_numbers_prg_to_ams = '452'
train_numbers_ams_to_prg = '453'

def search_availability(date_str: str, train_number: str) -> dict:
    body = {
        "trainRouteId": "77ab1c5a-ea0b-4634-7cdd-08db0daabe3f",
        "fromLocationId":"5457076",
        "toLocationId":"8400058",
        "passengerTypes":[72],
        "trainNumber":train_number,
        "travelDate":date_str,
        "bicycleCount":0
    }
    url = 'https://europeansleeperprod-api.azurewebsites.net/api/search/availability'
    response = requests.post(url, json=body)
    return response.json()

In [40]:
results = []
start_date = date.today()
for i in tqdm(range(90)):
    current_date = (start_date + timedelta(days=i)).strftime('%Y-%m-%d')
    results.append(search_availability(current_date))

2026-02-16
2026-02-17
2026-02-18
2026-02-19
2026-02-20
2026-02-21
2026-02-22
2026-02-23
2026-02-24
2026-02-25
2026-02-26
2026-02-27
2026-02-28
2026-03-01
2026-03-02
2026-03-03
2026-03-04
2026-03-05
2026-03-06
2026-03-07
2026-03-08
2026-03-09
2026-03-10
2026-03-11
2026-03-12
2026-03-13
2026-03-14
2026-03-15
2026-03-16
2026-03-17
2026-03-18
2026-03-19
2026-03-20
2026-03-21
2026-03-22
2026-03-23
2026-03-24
2026-03-25
2026-03-26
2026-03-27
2026-03-28
2026-03-29
2026-03-30
2026-03-31
2026-04-01
2026-04-02
2026-04-03
2026-04-04
2026-04-05
2026-04-06
2026-04-07
2026-04-08
2026-04-09
2026-04-10
2026-04-11
2026-04-12
2026-04-13
2026-04-14
2026-04-15
2026-04-16
2026-04-17
2026-04-18
2026-04-19
2026-04-20
2026-04-21
2026-04-22
2026-04-23
2026-04-24
2026-04-25
2026-04-26
2026-04-27
2026-04-28
2026-04-29
2026-04-30
2026-05-01
2026-05-02
2026-05-03
2026-05-04
2026-05-05
2026-05-06
2026-05-07
2026-05-08
2026-05-09
2026-05-10
2026-05-11
2026-05-12
2026-05-13
2026-05-14
2026-05-15
2026-05-16


In [42]:
rows = []

for result in results:
    availability = result['availabilityResult']['availability']
    if availability['departureStationName'] == 'Not available':
        continue

    row = {**availability}
    for price_class in row['priceClasses']:
        row[price_class['placeTypeKey']] = None if price_class['prices'] is None else price_class['prices']['eur']

    del row['priceClasses']
    del row['sections']

    rows.append(row)

In [43]:
df = pd.DataFrame(rows)

In [45]:
df.sort_values(by='couchette-5').head()

,requestedDepartureStationId,requestedArrivalStationId,departureStationId,departureStationName,departureTime,arrivalStationId,arrivalStationName,arrivalTime,seat-second-class,couchette-5,couchette-5-women-only,couchette-5-private,berth-single,comfort-single,berth-double,comfort-double,berth-triple,comfort-triple,berth-triple-women-only,comfort-triple-women-only
12,5457076,8400058,5457076,PRAHA HL. N.,2026-03-17T18:00:00,8400058,Amsterdam CS,2026-03-18T07:31:00,29.99,49.99,49.99,179.99,359.99,239.99,179.99,119.99,149.99,99.99,149.99,99.99
15,5457076,8400058,5457076,PRAHA HL. N.,2026-03-24T18:00:00,8400058,Amsterdam CS,2026-03-25T07:30:00,39.99,49.99,49.99,179.99,319.99,239.99,159.99,119.99,129.99,99.99,129.99,99.99
9,5457076,8400058,5457076,PRAHA HL. N.,2026-03-10T18:00:00,8400058,Amsterdam CS,2026-03-11T07:30:00,39.99,49.99,49.99,179.99,459.99,289.99,229.99,149.99,189.99,119.99,189.99,119.99
13,5457076,8400058,5457076,PRAHA HL. N.,2026-03-19T18:00:00,8400058,Amsterdam CS,2026-03-20T06:26:00,49.99,59.99,59.99,209.99,359.99,239.99,179.99,119.99,149.99,99.99,149.99,99.99
36,5457076,8400058,5457076,PRAHA HL. N.,2026-05-12T18:05:00,8400058,Amsterdam CS,2026-05-13T06:26:00,49.99,59.99,59.99,209.99,459.99,289.99,229.99,149.99,189.99,119.99,189.99,119.99
